<a href="https://colab.research.google.com/github/tejashwinirk/Agentic-AI/blob/main/Lab5_AgenicAi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install langchain \
langchain-community \
langchain-groq \
chromadb \
sentence-transformers \
pypdf


In [ ]:
from google.colab import files
from google.colab.userdata import get

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq

In [ ]:
groq_api_key = get("Agentic")

print("API Key Loaded Successfully")

API Key Loaded Successfully


In [ ]:
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded:", pdf_path)

Saving jkBms_User_manual.pdf to jkBms_User_manual.pdf
Uploaded: jkBms_User_manual.pdf


In [ ]:
loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Pages Loaded:", len(documents))

Pages Loaded: 30


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

print("Chunks:", len(docs))

Chunks: 58


In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_1357/914705057.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
db = Chroma.from_documents(
    docs,
    embedding
)

print("Vector Database Ready")

Vector Database Ready


In [ ]:
llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.1-8b-instant",
    temperature=0
)

print("LLM Connected")

LLM Connected


In [ ]:
def answer_from_pdf(query):

    retrieved_docs = db.as_retriever().invoke(query)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    prompt = f"""
You are answering ONLY from the provided PDF.

Context:
{context}

Question:
{query}

If the answer is not contained in the context,
reply exactly:

I don't know
"""

    return llm.invoke(prompt).content

In [ ]:
def adaptive_answer(question, max_tries=3):

    query = question

    for attempt in range(1, max_tries + 1):

        print("=" * 50)
        print(f"Attempt {attempt}")
        print("Query:", query)

        answer = answer_from_pdf(query)

        print("\nAnswer:")
        print(answer)

        if "i don't know" not in answer.lower():

            print("\nAnswer Found!")
            return answer

        print("\nRephrasing Query...\n")

        query = llm.invoke(
            f"Rephrase this search query differently while keeping the same meaning:\n{query}"
        ).content

    return "Could not find an answer after several attempts."

In [ ]:
question = "What is the main conclusion of the document?"

final_answer = adaptive_answer(question)

print("\n")
print("=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(final_answer)

Attempt 1
Query: What is the main conclusion of the document?

Answer:
I don't know

Rephrasing Query...

Attempt 2
Query: Here are a few rephrased search queries with the same meaning:

1. What is the overall summary of the document?
2. What is the key takeaway from the document?
3. What is the main point or finding of the document?
4. What is the author's central argument or conclusion?
5. What is the document's primary conclusion or recommendation?

These rephrased queries can help you find the same information as the original query, but with slightly different wording.

Answer:
1. The document is a user manual for a protection board, specifically discussing its short circuit protection function and how to set it up.

2. The key takeaway is that the protection board has a standard short circuit protection function that can be adjusted by the user through the APP.

3. The main point is that the user can increase the short circuit protection delay to prevent false triggers caused by h

In [ ]:
adaptive_answer("Who is the author?")

adaptive_answer("Summarize the document")

adaptive_answer("What methodology was used?")

adaptive_answer("What are the future recommendations?")

adaptive_answer("Explain the introduction")

Attempt 1
Query: Who is the author?

Answer:
I don't know

Rephrasing Query...

Attempt 2
Query: Here are a few rephrased search queries with the same meaning:

1. Who wrote this?
2. What is the author's name?
3. Who is the creator of this content?
4. Who penned this?
5. Who is the writer of this piece?

These rephrased queries can be used in various contexts, such as searching for information about a book, article, or any other written content.

Answer:
极极空空科科技有限公司
Jiikkonng TTechhnology Co., Ltd

Answer Found!
Attempt 1
Query: Summarize the document

Answer:
The document appears to be a maintenance manual for a lithium battery active balance protection board. It includes the following information:

- Product adaptation guide and function selection guides for 150A and 200A protection boards
- Function introduction and usage instructions, including active equalization, UART and display interface, heating function, and alarm function
- Device usage instructions, including APP installati

'The protection board is a small, simple, and full-featured device that can be widely used in various battery pack applications. It has a mobile APP supporting Android and IOS operating systems, which can be connected to the protection board through Bluetooth to check the battery working status, modify the working parameters, control the charge and discharge switch, and so on. The protection board supports active equalization technology, which can ensure the consistency of the battery to the greatest extent, improve the battery range, and delay the aging of the battery.'